| Feature                | FAISS   | ChromaDB |
| ---------------------- | ------- | -------- |
| Flat index             | ✅       | ❌        |
| IVF                    | ✅       | ❌        |
| HNSW                   | ✅       | ✅        |
| PQ                     | ✅       | ❌        |
| Manual index selection | ✅       | ❌        |
| Metadata filtering     | Limited | ✅        |
| Persistence            | Manual  | Built-in |
| LangChain integration  | ✅       | ✅        |



FAISS = vector index/search engine

You manually manage:
- index type
- dimension
- metric
- docstore
- ID mapping
- persistence
  
Chroma = vector database
It manages:
- vectors
- documents
- metadata
- IDs
- collections
- index
- persistence


FAISS is a low-level, high-performance library for dense-vector similarity search and clustering. It gives developers direct control over index structures such as Flat, IVF, HNSW and product-quantized indexes, and it offers strong CPU and GPU capabilities. However, FAISS is not a complete vector database: document storage, metadata management, filtering, CRUD APIs, collections, persistence orchestration and server infrastructure generally need to be handled separately.

Chroma is a retrieval database/search infrastructure designed for AI applications. It stores embeddings together with documents, metadata and IDs, and provides collections, persistence, metadata filtering, full-text and sparse retrieval, CRUD operations and client-server or hosted deployment options. In current Chroma, single-node vector search uses HNSW, while its broader schema and cloud architecture also support other retrieval indexes such as SPANN and sparse/full-text indexes.

Therefore, FAISS is preferable when low-level index control, custom ANN algorithms, compression or GPU optimization is the priority. Chroma is preferable when building a complete RAG application that needs database-style storage, filtering, updates and operational simplicity.

| Feature         | FAISS                 | Chroma |
| --------------- | --------------------- | ------ |
| Store vectors   | ✅                     | ✅      |
| Store documents | ❌                     | ✅      |
| Store metadata  | ❌                     | ✅      |
| Collections     | ❌                     | ✅      |
| CRUD            | Limited               | ✅      |
| Filtering       | ❌ (native)            | ✅      |
| Persistence     | Basic index save/load | ✅      |
| Client APIs     | ❌                     | ✅      |
| Server mode     | ❌                     | ✅      |


Tumhare code me FAISS ke saath document aur metadata dono store ho rahe the, but crucial point ye hai:

Unhe native FAISS store nahi kar raha tha; LangChain ka FAISS wrapper store kar raha tha.

LangChain FAISS VectorStore
│
├── FAISS index
│     └── Numerical embedding vectors
│
├── InMemoryDocstore
│     └── LangChain Document objects
│         ├── page_content
│         └── metadata
│
└── index_to_docstore_id
      └── FAISS position ko Document ID se map karta hai

vector_store = FAISS(
    embedding_function=embeddings,
    index=faiss_index,
    docstore=InMemoryDocstore(),
    index_to_docstore_id={}
)

Chroma me document, metadata, ID aur embedding same database collection ke records hain:

Chroma collection

collection.add(
    ids=["chunk-1"],
    documents=["Llama 2 is a family of language models."],
    metadatas=[
        {
            "source": "llama2.pdf",
            "page": 5
        }
    ],
    embeddings=[[0.1, 0.2, 0.3]]
)

| Component        | LangChain + FAISS             | Chroma                          |
| ---------------- | ----------------------------- | ------------------------------- |
| Embeddings       | Native FAISS index            | Chroma vector index             |
| Documents        | LangChain `Docstore`          | Chroma collection               |
| Metadata         | LangChain `Document.metadata` | Chroma collection record        |
| Mapping          | `index_to_docstore_id`        | Internally managed              |
| Save             | FAISS file + pickle           | Database persistence            |
| Metadata filters | Wrapper/application handling  | Native database filtering       |
| Collections      | Not native to FAISS           | Native                          |
| CRUD             | Wrapper/index-dependent       | Native record operations        |
| Server/cloud     | Separate system required      | Supported database architecture |


https://www.trychroma.com/

In [2]:
from dotenv import load_dotenv
import os

from langchain_google_genai import (
    GoogleGenerativeAIEmbeddings,
    ChatGoogleGenerativeAI
)

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

In [3]:
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
from langchain_openai import OpenAIEmbeddings
embeddings=OpenAIEmbeddings(model="text-embedding-3-large")

In [6]:
# --------------------------------------------------
# 3. Load PDF
# --------------------------------------------------
file_path = r"../data/llama2-research-paper.pdf"
loader = PyPDFLoader(file_path)
pages = loader.load()
print("Total pages:", len(pages))

Exceeded 5000 form XObject invocations while extracting text; further form content is skipped.


Total pages: 77


In [7]:
# --------------------------------------------------
# 4. Create chunks
# --------------------------------------------------
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=2000,
    chunk_overlap=200,
    separators=[
        "\n\n",
        "\n",
        ". ",
        " ",
        ""
    ]
)
chunks = text_splitter.split_documents(pages)
print("Total chunks:", len(chunks))

Total chunks: 175


In [9]:
# --------------------------------------------------
# 5. Create Chroma vector store
# --------------------------------------------------
vector_store = Chroma(
    collection_name="llama2_collection",
    embedding_function=embeddings,
    persist_directory="./chroma_db_llama2",
    collection_metadata={
        "hnsw:space": "cosine"
    }
)

In [10]:
# --------------------------------------------------
# 6. Add documents
# --------------------------------------------------

document_ids = vector_store.add_documents(
    documents=chunks
)

print("Documents added:", len(document_ids))

print(
    "Total documents stored:",
    vector_store._collection.count()
)


RateLimitError: Error code: 429 - {'error': {'message': 'You have no credits remaining. Add credits to continue using the API at https://platform.openai.com/settings/organization/billing/.', 'type': 'insufficient_quota', 'param': None, 'code': 'credit_balance_exhausted'}}

As we don't have openAI credits use below huggiginface

In [11]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

In [14]:
# --------------------------------------------------
# 5. Create Chroma vector store
# --------------------------------------------------
vector_store = Chroma(
    collection_name="llama2_collection_minilm",
    embedding_function=embeddings,
    persist_directory="./chroma_db_llama2_minilm",
    collection_metadata={"hnsw:space": "cosine"},
)

In [15]:
# --------------------------------------------------
# 6. Add documents
# --------------------------------------------------

document_ids = vector_store.add_documents(
    documents=chunks
)

print("Documents added:", len(document_ids))

print(
    "Total documents stored:",
    vector_store._collection.count()
)


Documents added: 175
Total documents stored: 175


In [24]:
document_ids

['bd966810-8cec-4311-ab17-9808a53a4a78',
 '6498c4cf-602f-4d6c-b015-e8dd9007cee5',
 '7de96c3c-2679-4b48-8f99-b9bbb8a10836',
 'ca4b9f87-944d-4ca0-95fc-457090da2dbc',
 '8e140080-075e-4892-9d9d-720cc7ec8494',
 '9dace571-a31b-4720-b7b3-d3100e67af7b',
 '4e1ec316-1368-4672-89dd-1cfc4333bafa',
 'ca0b5984-5dc3-4019-ab3b-2deba48bac75',
 '978f4a79-29ae-4c64-9bae-6acf3f11edce',
 'fce8944c-bc11-45be-887a-3b0aefd90d87',
 '89099975-b217-4fc7-8df9-7020029cb717',
 '14bc00a4-1a02-4229-9554-b21affc7b673',
 '4f9fef08-9446-4869-b029-9a1bb0d43643',
 'bf7d7c55-5040-464f-bbb1-567f8902ccc1',
 '3ff46173-e09d-4b2a-a692-c4949c4aba04',
 'f0b939a4-f485-4dfd-b455-157fab4d01a8',
 '714fb28b-f95c-43ca-aa1d-eeec9d1d8cbe',
 '2d2901c0-e94b-4eec-b99b-8917deb17d9d',
 '059fc299-8a98-4eae-a26d-52130257e2bb',
 '1ede4783-d7f7-40f9-af14-78ae936b8944',
 'ffcb56ea-5e1c-4dd5-a6a1-b073ec5901dd',
 '2ee78bf2-32b5-4ce6-b247-ed8f7412d64f',
 'db13e1d6-1567-4003-ab6a-ca78b121ad02',
 '6005d480-74ab-4623-b8ab-ac17d4885e92',
 '7749fe7f-9736-

In [17]:
# --------------------------------------------------
# 7. Create retriever
# --------------------------------------------------

retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": 5
    }
)

In [18]:
# --------------------------------------------------
# 8. Test retriever
# --------------------------------------------------

query = "What is the architecture of Llama 2?"

retrieved_documents = retriever.invoke(query)

for i, document in enumerate(
    retrieved_documents,
    start=1
):
    print(f"\n--- Retrieved document {i} ---")
    print(document.page_content[:500])
    print("Metadata:", document.metadata)



--- Retrieved document 1 ---
guide¶ and code examples‖ to facilitate the safe deployment ofLlama 2 and Llama 2-Chat. More details of
our responsible release strategy can be found in Section 5.3.
The remainder of this paper describes our pretraining methodology (Section 2), fine-tuning methodology
(Section 3), approach to model safety (Section 4), key observations and insights (Section 5), relevant related
work (Section 6), and conclusions (Section 7).
‡https://ai.meta.com/resources/models-and-libraries/llama/
§We are delayi
Metadata: {'producer': 'pdfTeX-1.40.25', 'page_label': '4', 'page': 3, 'subject': '', 'creationdate': '2023-07-20T00:30:36+00:00', 'moddate': '2023-07-20T00:30:36+00:00', 'title': '', 'creator': 'LaTeX with hyperref', 'trapped': '/False', 'keywords': '', 'total_pages': 77, 'author': '', 'source': '../data/llama2-research-paper.pdf', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5'}

--- Retrieved document 2

In [19]:
# --------------------------------------------------
# 9. Prompt
# --------------------------------------------------
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template(
    """
    You are a question-answering assistant.

    Answer the question only from the provided context.

    If the context does not contain the answer, say:
    "I do not have enough information in the provided document."

    Context:
    {context}

    Question:
    {question}

    Answer:
    """
)

In [20]:
# --------------------------------------------------
# 10. Format documents
# --------------------------------------------------

def format_docs(docs):
    return "\n\n".join(
        f"""
        Source: {doc.metadata.get("source")}
        Page: {doc.metadata.get("page")}

        {doc.page_content}
        """
        for doc in docs
    )

In [21]:
# --------------------------------------------------
# 11. LLM
# --------------------------------------------------
from langchain_google_genai import ChatGoogleGenerativeAI


model = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0
)

In [22]:
# --------------------------------------------------
# 12. RAG chain
# --------------------------------------------------
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt
    | model
    | StrOutputParser()
)

In [23]:
# --------------------------------------------------
# 13. Ask question
# --------------------------------------------------

answer = rag_chain.invoke(
    "What is the architecture of Llama 2?"
)

print("\nFinal answer:\n")
print(answer)

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.



Final answer:

Llama 2 is an auto-regressive language model that uses an optimized transformer architecture. It has an expanded context window from 2048 tokens to 4096 tokens and uses Grouped-Query Attention. The tuned versions use supervised fine-tuning (SFT) and reinforcement learning with human feedback (RLHF) to align to human preferences for helpfulness and safety.
